# Project setup

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "shared" / "config.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/sanjeet/ai_workspace/KWS model/sih-2026


# Load configuration

In [2]:
from shared.config import KEYWORD, MODELS_DIR
from shared.feature_spec import SPEC, FEATURE_SHAPE
from training.model import N_CLASSES

print(f"Keyword       : {KEYWORD}")
print(f"Feature shape : {FEATURE_SHAPE}")
print(f"Classes       : {N_CLASSES}")

I0000 00:00:1789894061.261691   42682 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Keyword       : marvin
Feature shape : (49, 20, 1)
Classes       : 3


# Load current validation set

In [3]:
from training.train import collect

train_data, val_data, speakers, gsc_split = collect(
    KEYWORD,
    holdout_speaker=None,
    noise_cap=400,
)

x_val, y_val, _ = val_data.arrays()

print(f"Validation samples : {len(x_val)}")
print(f"Feature shape      : {x_val.shape[1:]}")
print(f"Using GSC test split: {gsc_split}")

  loaded 2000 clips...
  loaded 3000 clips...
  loaded 4000 clips...
  loaded 6000 clips...
  loaded 7000 clips...
  loaded 8000 clips...
  loaded 9000 clips...
  loaded 10000 clips...
  loaded 11000 clips...
  loaded 12000 clips...
  loaded 13000 clips...
  loaded 14000 clips...
  loaded 15000 clips...
  loaded 16000 clips...
  loaded 17000 clips...
  loaded 18000 clips...
  loaded 19000 clips...
  loaded 20000 clips...
  loaded 21000 clips...
  loaded 22000 clips...
  loaded 23000 clips...
Validation samples : 11164
Feature shape      : (49, 20, 1)
Using GSC test split: True


# Locate the trained model

In [4]:
model_candidates = [
    MODELS_DIR / f"{KEYWORD}_best.keras",
    MODELS_DIR / f"{KEYWORD}.keras",
    MODELS_DIR / f"{KEYWORD}_final.keras",
]

model_path = next(
    (p for p in model_candidates if p.exists()),
    None,
)

if model_path is None:
    raise FileNotFoundError(
        f"No trained model found in {MODELS_DIR}"
    )

print(f"Using model: {model_path}")

Using model: /home/sanjeet/ai_workspace/KWS model/sih-2026/models/marvin.keras


# Load model

In [5]:
import tensorflow as tf

model = tf.keras.models.load_model(
    model_path,
    compile=False,
)

model.summary()

W0000 00:00:1789894122.727420   42682 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "dscnn_s_residual"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mfcc (InputLayer)   │ (None, 49, 20, 1) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 25, 10,    │      2,560 │ mfcc[0][0]        │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 25, 10,    │        256 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_relu (ReLU)    │ (None, 25, 10,    │          0 │ stem_bn[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pad          │ (None, 27, 12,    │          0 │ stem_relu[0][0]   │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_dwconv       │ (None, 25, 10,    │        576 │ block1_pad[0][0]  │
│ (DepthwiseConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_dw_bn        │ (None, 25, 10,    │        256 │ block1_dwconv[0]… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_dw_relu      │ (None, 25, 10,    │          0 │ block1_dw_bn[0][… │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pwconv       │ (None, 25, 10,    │      4,096 │ block1_dw_relu[0… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_pw_bn        │ (None, 25, 10,    │        256 │ block1_pwconv[0]… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_add (Add)    │ (None, 25, 10,    │          0 │ block1_pw_bn[0][… │
│                     │ 64)               │            │ stem_relu[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1_out (ReLU)   │ (None, 25, 10,    │          0 │ block1_add[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pad          │ (None, 27, 12,    │          0 │ block1_out[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_dwconv       │ (None, 25, 10,    │        576 │ block2_pad[0][0]  │
│ (DepthwiseConv2D)   │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_dw_bn        │ (None, 25, 10,    │        256 │ block2_dwconv[0]… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_dw_relu      │ (None, 25, 10,    │          0 │ block2_dw_bn[0][… │
│ (ReLU)              │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block2_pwconv       │ (None, 25, 10,    │      4,096 │ block2_dw_relu[0… │
│ (Conv2D)            │ 64)               │            │                 

 Total params: 23,747 (92.76 KB)

 Trainable params: 22,595 (88.26 KB)

 Non-trainable params: 1,152 (4.50 KB)

In [6]:
import tensorflow as tf
from pathlib import Path

model_path = PROJECT_ROOT / "models" / "marvin.keras"
models_dir = PROJECT_ROOT / "models"

# ---------- FP16 ----------
fp16_converter = tf.lite.TFLiteConverter.from_keras_model(
    tf.keras.models.load_model(model_path, compile=False)
)

fp16_converter.optimizations = [tf.lite.Optimize.DEFAULT]
fp16_converter.target_spec.supported_types = [tf.float16]

fp16_model = fp16_converter.convert()

fp16_path = models_dir / "marvin.fp16.tflite"
fp16_path.write_bytes(fp16_model)


# ---------- INT8 ----------
int8_converter = tf.lite.TFLiteConverter.from_keras_model(
    tf.keras.models.load_model(model_path, compile=False)
)

int8_converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset for activation calibration
def representative_dataset():
    for i in range(min(len(x_val), 500)):
        yield [x_val[i:i+1].astype("float32")]

int8_converter.representative_dataset = representative_dataset

int8_converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

int8_converter.inference_input_type = tf.int8
int8_converter.inference_output_type = tf.int8

int8_model = int8_converter.convert()

int8_path = models_dir / "marvin.int8.tflite"
int8_path.write_bytes(int8_model)


print(f"FP16: {fp16_path} — {len(fp16_model):,} bytes")
print(f"INT8: {int8_path} — {len(int8_model):,} bytes")

INFO:tensorflow:Assets written to: /tmp/tmpzrkhpsqd/assets


INFO:tensorflow:Assets written to: /tmp/tmpzrkhpsqd/assets


Saved artifact at '/tmp/tmpzrkhpsqd'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 20, 1), dtype=tf.float32, name='mfcc')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  127996379897616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379897808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379896272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379898384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379898576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379896656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379898000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379898192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379897424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379899536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996379898960: TensorS

W0000 00:00:1789894124.756460   42682 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1789894124.756514   42682 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1789894124.756969   42682 reader.cc:83] Reading SavedModel from: /tmp/tmpzrkhpsqd
I0000 00:00:1789894124.759318   42682 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1789894124.759331   42682 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpzrkhpsqd
I0000 00:00:1789894124.787890   42682 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1789894124.792094   42682 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1789894124.895524   42682 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpzrkhpsqd
I0000 00:00:1789894124.923576   42682 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 166617 microseconds.
I0000 00:00:1789894124.956343   4268

INFO:tensorflow:Assets written to: /tmp/tmphozq_g1d/assets


INFO:tensorflow:Assets written to: /tmp/tmphozq_g1d/assets


Saved artifact at '/tmp/tmphozq_g1d'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 20, 1), dtype=tf.float32, name='mfcc')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  127996737764048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737762128: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737765200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737755984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737756752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737763856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737762512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737762320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737764240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737763472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127996737764816: TensorS

/home/sanjeet/ai_workspace/KWS model/sih-2026/.venv/lib/python3.12/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1789894126.647293   42682 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1789894126.647356   42682 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1789894126.647602   42682 reader.cc:83] Reading SavedModel from: /tmp/tmphozq_g1d
I0000 00:00:1789894126.649519   42682 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1789894126.649543   42682 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmphozq_g1d
I0000 00:00:1789894126.674622   42682 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1789894126.778784   42682 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmphozq_g1d
I0000 00:00:1789894126.804825   42682 loader.cc:471] 

FP16: /home/sanjeet/ai_workspace/KWS model/sih-2026/models/marvin.fp16.tflite — 59,072 bytes
INT8: /home/sanjeet/ai_workspace/KWS model/sih-2026/models/marvin.int8.tflite — 48,888 bytes


fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1789894127.757482   42682 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.
